In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "scripts")
print(os.getcwd())

b:\TrailGuard-1\trailguard\tests


In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score, roc_auc_score
from builder_lanl_b2 import build_from_files, WINDOW_SECONDS
from unsupervised_helper_b2 import train_unsupervised, get_normalized_unsup_score

In [3]:
USE_TOY = True      # change to False when the full run has finished

if USE_TOY:
    AUTH, RED = "data/raw/toy_subset.csv.gz", "data/raw/toy_redteam_hashed.csv.gz"
    OUT_CSV, OUT_JSON = "data/processed/toy_features.csv.gz", "data/processed/toy_columns.json"
else:
    AUTH, RED = "data/raw/lanl_subset_auth.csv.gz", "data/raw/lanl_redteam_hashed.csv.gz"
    OUT_CSV, OUT_JSON = "data/processed/features_lanl.csv.gz", "data/processed/lanl_feature_columns.json"

In [4]:
feat = build_from_files(AUTH, RED, OUT_CSV, OUT_JSON)
print(feat.shape, "| positives:", int(feat.label.sum()), "| rate: %.4f" % feat.label.mean())
feat.head()

(19228, 15) | positives: 37 | rate: 0.0019


,time_key,user_hash,label,total_auth_events,fail_auth_count,fail_auth_ratio,off_hours_event_ratio,unique_src_computers,unique_dst_computers,new_dst_computer_count,network_logon_ratio,kerberos_auth_ratio,total_auth_events_z,unique_dst_computers_z,fail_auth_count_z
0,8,04126a6866e27b7a,0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.000000,0.0,0.0,0.0
1,8,0fada21f257e2099,0,3.0,0.0,0.0,0.0,1.0,2.0,2.0,1.0,0.666667,0.0,0.0,0.0
2,8,141a3e4dc6e70b0f,0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.000000,0.0,0.0,0.0
3,8,143aad04c2ece463,0,3.0,0.0,0.0,0.0,1.0,2.0,2.0,1.0,0.666667,0.0,0.0,0.0
4,8,14eee693117a2be2,0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.000000,0.0,0.0,0.0


In [5]:
def temporal_split(df):
    df = df.sort_values("time_key", kind="stable").reset_index(drop=True)
    n = len(df); a, b = int(n * 0.70), int(n * 0.85)
    return df.iloc[:a], df.iloc[a:b], df.iloc[b:]

def prepare(df):
    tr, va, te = temporal_split(df)
    cols = [c for c in df.columns if c not in ("time_key", "user_hash", "label")]
    imp = SimpleImputer(strategy="median").fit(tr[cols])          # fit on TRAIN only
    sc = RobustScaler().fit(imp.transform(tr[cols]))
    f = lambda d: sc.transform(imp.transform(d[cols]))
    out = dict(cols=cols, imp=imp, sc=sc,
               Xtr=f(tr), Xva=f(va), Xte=f(te),
               ytr=tr.label.values, yva=va.label.values, yte=te.label.values,
               tk_va=va.time_key.values, tk_te=te.time_key.values)
    for name, d in [("train", tr), ("val", va), ("test", te)]:
        print(f"{name:5s} rows={len(d):8d} positives={int(d.label.sum()):5d}")
    return out

def metrics(y, s):
    if y.sum() == 0:
        return dict(pr_auc=np.nan, roc_auc=np.nan)
    return dict(pr_auc=average_precision_score(y, s), roc_auc=roc_auc_score(y, s))

def budget_metrics(time_key, y, score, ks=(5, 10, 20)):
    day = (time_key * WINDOW_SECONDS) // 86400
    d = pd.DataFrame(dict(day=day, y=y, s=score))
    res = {}
    for k in ks:
        sel = d.sort_values("s", ascending=False).groupby("day").head(k)
        res[f"P@{k}/day"] = round(sel.y.sum() / max(len(sel), 1), 3)
        res[f"R@{k}/day"] = round(sel.y.sum() / max(d.y.sum(), 1), 3)
    return res

In [6]:
D = prepare(feat)
if D["ytr"].sum() == 0:
    print("NOTE: no attacks in Train. Known limitation when red-team activity is clustered in time.")

train rows=   13459 positives=    0
val   rows=    2884 positives=   18
test  rows=    2885 positives=   19
NOTE: no attacks in Train. Known limitation when red-team activity is clustered in time.


In [7]:
results = []
for n_est in (100, 200, 300):
    for ms in ("auto", 256, 1024):
        m = IsolationForest(n_estimators=n_est, max_samples=ms, contamination="auto",
                            random_state=42, n_jobs=-1)
        m.fit(D["Xtr"][D["ytr"] == 0])
        s = -m.score_samples(D["Xva"])
        results.append(dict(model="IsolationForest", n_estimators=n_est, max_samples=ms,
                            **metrics(D["yva"], s)))

lof = LocalOutlierFactor(n_neighbors=35, novelty=True).fit(D["Xtr"][D["ytr"] == 0])
results.append(dict(model="LOF(novelty)", n_estimators=None, max_samples=None,
                    **metrics(D["yva"], -lof.score_samples(D["Xva"]))))

zmax = np.abs(D["Xva"]).max(axis=1)
results.append(dict(model="Max |Z| baseline", n_estimators=None, max_samples=None,
                    **metrics(D["yva"], zmax)))

res = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
res

,model,n_estimators,max_samples,pr_auc,roc_auc
8,IsolationForest,300.0,1024,1.000000,1.000000
1,IsolationForest,100.0,256,1.000000,1.000000
0,IsolationForest,100.0,auto,1.000000,1.000000
2,IsolationForest,100.0,1024,1.000000,1.000000
3,IsolationForest,200.0,auto,1.000000,1.000000
5,IsolationForest,200.0,1024,1.000000,1.000000
4,IsolationForest,200.0,256,1.000000,1.000000
6,IsolationForest,300.0,auto,1.000000,1.000000
7,IsolationForest,300.0,256,1.000000,1.000000
10,Max |Z| baseline,NaN,None,0.653169,0.982767


In [ ]:
best = res[res.model == "IsolationForest"].iloc[0]
ms = best.max_samples if best.max_samples == "auto" else int(best.max_samples)
model, min_s, max_s = train_unsupervised(D["Xtr"], D["ytr"], n_estimators=int(best.n_estimators), max_samples=ms)
print("chosen:", int(best.n_estimators), ms, "| min_s=%.4f max_s=%.4f" % (min_s, max_s))

test_score = get_normalized_unsup_score(model, D["Xte"], min_s, max_s)
assert 0.0 <= test_score.min() and test_score.max() <= 1.0

print("TEST:", metrics(D["yte"], test_score))
print("TEST alert budget:", budget_metrics(D["tk_te"], D["yte"], test_score))
print("Positives are very sparse, so treat these numbers as indicative, not conclusive.")

chosen: 300 1024 | min_s=0.3453 max_s=0.6467
TEST: {'pr_auc': 0.37254901960784315, 'roc_auc': 0.994417306350314}
TEST alert budget: {'P@5/day': np.float64(0.4), 'R@5/day': np.float64(0.526), 'P@10/day': np.float64(0.38), 'R@10/day': np.float64(1.0), 'P@20/day': np.float64(0.19), 'R@20/day': np.float64(1.0)}
Positives are very sparse, so treat these numbers as indicative, not conclusive.
